# Ropedia Academy — A6 · Rotations & motion representations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/A6.ipynb)

> **Implements the continuous 6D→matrix fix and verifies it yields a valid rotation, then shows local-vs-global frame composition along the kinematic chain.**
>
> 实现连续的 6D→矩阵修复并验证它给出有效旋转，再演示沿运动学链的局部 vs 全局坐标系组合。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/A6

In [ ]:
import torch, torch.nn.functional as F

# How you ENCODE rotation decides if a net can learn it. Euler -> gimbal lock;
# quaternion -> double-cover sign flips; ANY <=4D encoding is discontinuous.
# Fix: regress a continuous 6D vector, Gram-Schmidt it into a rotation matrix.
def sixd_to_R(d6):                              # network outputs 6 numbers
    a1, a2 = d6[..., :3], d6[..., 3:]
    b1 = F.normalize(a1, dim=-1)
    b2 = F.normalize(a2 - (b1*a2).sum(-1, keepdim=True)*b1, dim=-1)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

R = sixd_to_R(torch.randn(6))
I = torch.eye(3)
print("valid rotation? orthonormal:", torch.allclose(R @ R.T, I, atol=1e-5),
      "det=+1:", bool(torch.allclose(R.det(), torch.tensor(1.), atol=1e-5)))

# Local (parent-relative) frames compose along the kinematic chain, so the same
# gesture is invariant to overall body orientation.
parent, local = sixd_to_R(torch.randn(6)), sixd_to_R(torch.randn(6))
child_global = parent @ local
print("child global orientation = parent ∘ local:", tuple(child_global.shape))

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/A6
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks